<a href="https://colab.research.google.com/github/kinchittrivedi/Kaggle/blob/main/Cleaning_Text_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
uciml_sms_spam_collection_dataset_path = kagglehub.dataset_download('uciml/sms-spam-collection-dataset')

print('Data source import complete.')


Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Data source import complete.


# Cleaning Text Data

**Welcome to your first practical step in Natural Language Processing (NLP)!** Most machine learning workflows begin with neat, **tabular data**, consisting of rows of independent observations and columns of structured numbers or categories. However, an immense amount of the world's data is trapped in a completely unstructured format: **raw text**.

> *For example, a messy string like:* `"URGENT! Call 09061701461 to claim your FREE prize!!"`

In this notebook, we will learn how to bridge the gap between messy text strings and powerful, tabular-based machine learning algorithms.

We will walk through the NLP workflow by analyzing the **SMS Spam Collection Dataset**. Our ultimate goal is to learn how to classify whether incoming SMS messages are malicious **spam** or legitimate (often referred to as **"ham"**).

## Understanding the Data: Exploring the SMS Spam Collection

The SMS Spam Collection is a public set of SMS labeled messages that have been collected for mobile phone spam research. When we load the raw file, you will notice it contains a few extra unnamed columns filled with missing values. We will focus entirely on the first two core columns:

* **`v1` (Our Target Label):** This column classifies each message as either **"ham"** (a legitimate, normal text message) or **"spam"** (an unwanted or malicious advertisement).
* **`v2` (Our Feature):** This column contains the raw, unstructured text string of the message itself.

Now, let's load the data.

In [3]:
import numpy as np
import pandas as pd
import os

# Load the SMS Spam Collection dataset into a pandas DataFrame
# We must specify encoding='latin-1' to handle the non-standard characters present in this specific text file
df = pd.read_csv(os.path.join(uciml_sms_spam_collection_dataset_path, 'spam.csv'), encoding='latin-1')

# Display the raw dataframe to see its initial structure and the extra unnamed columns
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


## Initial Data Cleaning

Before we can start cleaning the text itself, we need to clean our DataFrame's structure. As you may have noticed when loading the raw data, the dataset contains several unnamed columns filled with missing values (`NaN`). These provide no value to our analysis.

Our first step is to discard those useless columns and isolate the two core pieces of information we actually need: the target and the text. To make our code clean and intuitive moving forward, we will also rename the default `v1` and `v2` headers to `label` and `message`.

In [4]:
# Isolate the target (v1) and the feature (v2) columns, dropping the unnamed NaN columns
df = df[['v1', 'v2']]

# Rename the columns to make them intuitive and highly readable
df.columns = ['label', 'message']

# Display the first 5 rows of our cleaned DataFrame structure
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Checking for Missing Data

Now that our columns are properly named, we must ensure the dataset is complete. In text analysis, a missing value (a completely blank message) can cause errors later down the pipeline when we attempt to split the text into tokens or convert it into a numerical matrix. We will check to see if there are any null values in our DataFrame.

As the output below shows, **we have zero missing values**.

In [5]:
# Check for missing values in both the 'label' and 'message' columns
df.isnull().sum()

,0
label,0
message,0


## Checking for Duplicates

While we do not have missing data, text datasets frequently contain duplicate entries. In SMS data, this often happens with automated spam blasts or generic system messages. We will check to see how many duplicate rows are hiding in our dataset.

As the output below shows, **we have 403 duplicate messages**.


In [6]:
# Check the total number of duplicate rows in the dataset
df.duplicated().sum()

np.int64(403)

## Dropping Duplicates and Checking Class Balance

To ensure our model learns the true underlying patterns of spam rather than simply memorizing repeated texts, **we will drop these duplicate rows, keeping only the first occurrence.** Earlier, our dataset had 5572 rows. As we can see from the shape output below, the 403 duplicates have been removed, leaving us with exactly 5169 unique text messages.

Next, we will check the distribution of our target labels. In spam detection, it is very common to have a severe **class imbalance**, which simply means **one category heavily outnumbers the other.** When we look at our remaining data, we find that we have 4,516 legitimate normal messages compared to only 653 spam messages.

This heavily skewed ratio makes perfect sense because most people naturally receive far more genuine texts than junk. However, understanding this massive difference right now is absolutely critical. **If we ignore this imbalance, our future model might take the easy way out and simply guess that every single message is normal.** Recognizing these exact numbers today will dictate exactly how we prepare our algorithm and evaluate its true performance later in this Learn Guide.

In [7]:
# Drop the duplicate rows, keeping only the first instance of each message
df = df.drop_duplicates(keep='first')

# Verify the new shape of the dataset to confirm duplicates were removed
print("--- New Dataset Shape ---")
print(df.shape)

# Inspect the distribution of our target labels to check for class imbalance
print("\n--- Class Distribution ---")
print(df['label'].value_counts())

--- New Dataset Shape ---
(5169, 2)

--- Class Distribution ---
label
ham     4516
spam     653
Name: count, dtype: int64


## Lowercasing

Now we can begin our actual text preprocessing. **To a computer, the words "Free", "FREE", and "free" are completely different features because algorithms are case-sensitive.** By converting all text to lowercase, we standardize our vocabulary. This prevents our model from learning redundant variations of the exact same word and significantly reduces the total number of unique words our model has to process.

In [8]:
# Convert all characters in the 'message' column to lowercase
df['message'] = df['message'].str.lower()

# Display the first 5 rows to verify all text is now lowercase
df.head()

,label,message
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."


## Removing Punctuation

Just like capital letters, punctuation marks can trick our model into treating the same word as multiple different features (for example, "winner" and "winner!" would be seen as two distinct words).

For this foundational NLP pipeline, we will strip out all punctuation. We achieve this using Python's `re` (Regular Expression) library. **We will apply a function that looks for any character that is not a standard alphanumeric word character or a space, and replaces it with nothing.**

In [9]:
#Remove Punctuation

import re

# Use regular expressions to remove all punctuation
# r'[^\w\s]' matches anything that is NOT an alphanumeric character (\w) or whitespace (\s)
df['message'] = df['message'].apply(lambda x: re.sub(r'[^\w\s]', '', x))


## Tokenization

Machine learning models cannot read continuous sentences; they need text broken down into smaller, discrete units. **Tokenization is the process of splitting a continuous string of text into a list of individual words (or tokens).**

*For example, the message `"win free tickets"` becomes the list `['win', 'free', 'tickets']`.*

We will use the Natural Language Toolkit (`nltk`) library to tokenize our messages, transforming each string into a structured list of words that our algorithms can eventually count and analyze.

In [11]:
import nltk
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

# Apply the word_tokenize function to split each message into a list of individual words
df['tokens'] = df['message'].apply(word_tokenize)

# Display the original message alongside the new list of tokens to see the transformation
print(df[['message', 'tokens']].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                             message  \
0  go until jurong point crazy available only in ...   
1                            ok lar joking wif u oni   
2  free entry in 2 a wkly comp to win fa cup fina...   
3        u dun say so early hor u c already then say   
4  nah i dont think he goes to usf he lives aroun...   

                                              tokens  
0  [go, until, jurong, point, crazy, available, o...  
1                     [ok, lar, joking, wif, u, oni]  
2  [free, entry, in, 2, a, wkly, comp, to, win, f...  
3  [u, dun, say, so, early, hor, u, c, already, t...  
4  [nah, i, dont, think, he, goes, to, usf, he, l...  


## Removing Stopwords

Stopwords are very common words like "the", "is", "in", and "and". While essential for human grammar, **they carry almost no predictive value for determining if a message is spam.** By filtering out these words, we reduce noise and help the model focus entirely on the meaningful keywords in the text.

In [12]:
import nltk
from nltk.corpus import stopwords

# Download the standard list of English stopwords from NLTK
nltk.download('stopwords')

# Store the English stopwords in a set for much faster processing
stop_words = set(stopwords.words('english'))

# Apply a filter to keep only the tokens that are NOT in our stop_words list
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

# Display the first 5 rows to see the filtered token lists
print(df['tokens'].head())

0    [go, jurong, point, crazy, available, bugis, n...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, 2, wkly, comp, win, fa, cup, fin...
3        [u, dun, say, early, hor, u, c, already, say]
4    [nah, dont, think, goes, usf, lives, around, t...
Name: tokens, dtype: object


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Stemming vs. Lemmatization

In natural language processing, words often appear in multiple grammatical forms (e.g., "run", "running", "ran"). To help our machine learning model recognize that these variations represent the exact same underlying concept, we reduce words back to their base form. There are two primary techniques for this:

* **Stemming:** This is a fast, rule-based approach that simply chops off the ends of words using rigid rules. For example, "caring" becomes "car", and "history" becomes "hist". While quick, it often results in incomplete non-words.
  
* **Lemmatization:** This is a smarter, dictionary-based approach. It analyzes the word's structure and context to return its proper linguistic root (called the "lemma"). For example, "caring" becomes "care", and "better" becomes "good".

![](https://github.com/jeffpoulshaju/Model-evaluation/raw/23a1c0493bdd429b80468de4bbe87c90672faca8/Stemming%20v%20lemmetization.png)

For this fast, beginner-friendly spam filter, **we will proceed with Stemming.** While it might leave us with some funny-looking root words (like turning "crazy" into "crazi"), our machine learning model doesn't need perfect English. It just needs consistent patterns.

In [13]:
# Stemming
from nltk.stem import PorterStemmer

ps = PorterStemmer()

# Apply the stemmer to chop off prefixes/suffixes and find the root stem
df['tokens'] = df['tokens'].apply(
    lambda words: [ps.stem(word) for word in words]
)

df.head()

,label,message,tokens
0,ham,go until jurong point crazy available only in ...,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,ok lar joking wif u oni,"[ok, lar, joke, wif, u, oni]"
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,u dun say so early hor u c already then say,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, goe, usf, live, around, tho..."


In [14]:
# Display the first 5 rows to verify the effect of stemming on the tokens
df.head()

,label,message,tokens
0,ham,go until jurong point crazy available only in ...,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,ok lar joking wif u oni,"[ok, lar, joke, wif, u, oni]"
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,u dun say so early hor u c already then say,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, goe, usf, live, around, tho..."


## Rejoining Tokens into Text

We have successfully cleaned our messages: we made everything lowercase, stripped out the punctuation, removed useless stopwords, and reduced words to their stems.

Right now, our messages are stored as lists of individual words (tokens). However, the machine learning tools we will use in the next notebook (like `CountVectorizer` and `TF-IDF`) expect to read continuous strings of text, not lists.

Our final step in this notebook is to stitch these cleaned tokens back together into a single, clean sentence. We will use Python's `.join()` function to combine the words, separated by a single space.

In [15]:
#Join tokens back into text
df['processed_message'] = df['tokens'].apply(lambda x: " ".join(x))
df[['tokens','processed_message']].head()

,tokens,processed_message
0,"[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...
1,"[ok, lar, joke, wif, u, oni]",ok lar joke wif u oni
2,"[free, entri, 2, wkli, comp, win, fa, cup, fin...",free entri 2 wkli comp win fa cup final tkt 21...
3,"[u, dun, say, earli, hor, u, c, alreadi, say]",u dun say earli hor u c alreadi say
4,"[nah, dont, think, goe, usf, live, around, tho...",nah dont think goe usf live around though


In [16]:
# --- SAVE THE CLEANED DATA ---

# Save our final dataframe to a CSV file.
# We set index=False so Pandas doesn't create a useless column of row numbers!
df.to_csv('cleaned_spam_data.csv', index=False)

print("Cleaned data saved as 'cleaned_spam_data.csv'")

Cleaned data saved as 'cleaned_spam_data.csv'


## What's Next?

Our text is officially cleaned, standardized, and saved! Now it's time to turn these words into numbers through a process called **Feature Extraction**.

Ready? Let's jump into the next step: [Feature Extraction](https://www.kaggle.com/code/jeffpoulshaju/feature-extraction)